# Results figures — step by step

This notebook rebuilds the final figures of the thesis from the 32 raw COMSOL
exports. The physics is already validated; here we focus on **turning the
eigenvalue tables into the figures that tell the story**.

**Roadmap**

1. Load the exports and inspect their provenance.
2. Sort the eigenvalues into modes (`VP_1` = ground state, ...).
3. Plot the uniform-chain spectrum vs impurity position.
4. Plot the non-uniform-chain spectrum vs `R_5`.
5. Plot energy and induced dipole vs electric field.

> **Data:** `../data/` · **Figures:** `../figures/` · **Helpers:** `chain_utils.py`
>
> The committed notebook has no outputs; run it to regenerate every PNG.
>
> **Note on language:** the figures keep the original Spanish labels because
> they are the published thesis figures. The documentation is in English.


## Environment

The original working notebook used `%matplotlib qt` for interactive windows.
In this repository the plots render inline (or with any Jupyter backend); no
change to the physics or the figure code is needed.


## Step 1 — imports and global style

The style block is the one used for the thesis: DejaVu Sans, semibold titles,
and readable tick labels.


## Step 2 — load the 32 COMSOL exports

Every file has **8 header lines** (`% Model`, `% Version`, `% Date`, ...) followed
by two columns: the sweep variable `X` and the eigenvalue in eV.

Units of `X`:

| Family | Meaning | Unit |
|---|---|---|
| `caso*.txt`, `EvsxiR5=*.txt` | impurity position `x_i` | m (converted to nm below) |
| `EvsR5*.txt` | inter-dot distance `R_5` | m (converted to nm below) |
| `c*_F.txt`, `AEvsF*.txt` | electric field `F` | kV/cm |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titleweight": "semibold",
    "axes.labelweight": "normal",
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 14
})
print("Librerías OK")

## Step 3 — sort eigenvalues into modes

COMSOL returns an **unordered list of eigenvalues** for every sweep point.
Sorting by `(X, eigenvalue)` and counting with `cumcount` guarantees that
`VP_1` is always the ground state, `VP_2` the first excited state, and so on.
Without this step the curves would cross and the figure would be meaningless.


In [ ]:
DATA = "../data/"   # COMSOL exports

def leer(nombre):
    return pd.read_csv(DATA + nombre, skiprows=8, sep=r'\s+',
                       names=['X', 'Valor_propio'], engine='python')

# --- Cadena lineal: E vs xi ---
df1   = leer('caso1.txt')
df2_2 = leer('caso2_2.txt')
df3   = leer('caso3.txt')

# --- Cadena lineal: E vs F ---
df1F   = leer('c1_F.txt')
df2_2F = leer('c2_2_F.txt')
df3F   = leer('c3_F.txt')

# --- Cadena no lineal: E vs R5 ---
# sin impureza
df_R5sind0   = leer('EvsR5sind0.txt')
df_R5sind10  = leer('EvsR5sind10.txt')
df_R5sind_10 = leer('EvsR5sind-10.txt')
# impureza en xi=0
df_R5xi0d0   = leer('EvsR5xi=0d0.txt')
df_R5xi0d10  = leer('EvsR5xi=0d10.txt')
df_R5xi0d_10 = leer('EvsR5xi=0d-10.txt')
# impureza en xi=R1
df_R5xiR1d0   = leer('EvsR5xi=R1d0.txt')
df_R5xiR1d10  = leer('EvsR5xi=R1d10.txt')
df_R5xiR1d_10 = leer('EvsR5xi=R1d-10.txt')
# impureza en xi=R5
df_R5xiR5d0   = leer('EvsR5xi=R5d0.txt')
df_R5xiR5d10  = leer('EvsR5xi=R5d10.txt')
df_R5xiR5d_10 = leer('EvsR5xi=R5d-10.txt')

# --- Cadena no lineal: E vs xi (R5 fijo) ---
df_xiR5_10d0   = leer('EvsxiR5=10d0.txt')
df_xiR5_10d10  = leer('EvsxiR5=10d10.txt')
df_xiR5_10d_10 = leer('EvsxiR5=10d-10.txt')
df_xiR5_50d0   = leer('EvsxiR5=50d0.txt')
df_xiR5_50d10  = leer('EvsxiR5=50d10.txt')
df_xiR5_50d_10 = leer('EvsxiR5=50d-10.txt')

# --- Cadena asimetrica: E vs F ---
df_AEvsF_R10_d_0      = leer('AEvsF_R10_d_0.txt')
df_AEvsF_R10_d_10     = leer('AEvsF_R10_d_10.txt')
df_AEvsF_R10_d_menos10= leer('AEvsF_R10_d_-10.txt')
df_AEvsF_R50_d_0      = leer('AEvsF_R50_d_0.txt')
df_AEvsF_R50_d_10     = leer('AEvsF_R50_d_10.txt')
df_AEvsF_R50_d_menos10= leer('AEvsF_R50_d_-10.txt')

print("Datos cargados OK")

## Step 4 — plotting helpers

`estilizar` removes the grid and fixes the spines; `fmt_mev` prints the
energy axis in meV without trailing zeros. `recortar_orbital` and
`recortar_puntos` crop the auxiliary orbital images used as insets.


In [ ]:
def organizar_valores_propios(df):
    df = df.copy()
    df = df.sort_values(by=['X', 'Valor_propio']).reset_index(drop=True)
    df['Modo'] = df.groupby('X').cumcount()
    df = df.pivot(index='X', columns='Modo', values='Valor_propio')
    df.columns = [f'VP_{i+1}' for i in df.columns]
    return df.reset_index()

In [ ]:
# Cadena lineal
df1o        = organizar_valores_propios(df1)
df2_2o      = organizar_valores_propios(df2_2)
df3o        = organizar_valores_propios(df3)

# Cadena lineal: E vs F
df1Fo   = organizar_valores_propios(df1F)
df2_2Fo = organizar_valores_propios(df2_2F)
df3Fo   = organizar_valores_propios(df3F)

# Cadena no lineal: E vs R5
df_R5sind0o   = organizar_valores_propios(df_R5sind0)
df_R5sind10o  = organizar_valores_propios(df_R5sind10)
df_R5sind_10o = organizar_valores_propios(df_R5sind_10)
df_R5xi0d0o    = organizar_valores_propios(df_R5xi0d0)
df_R5xi0d10o   = organizar_valores_propios(df_R5xi0d10)
df_R5xi0d_10o  = organizar_valores_propios(df_R5xi0d_10)
df_R5xiR1d0o   = organizar_valores_propios(df_R5xiR1d0)
df_R5xiR1d10o  = organizar_valores_propios(df_R5xiR1d10)
df_R5xiR1d_10o = organizar_valores_propios(df_R5xiR1d_10)
df_R5xiR5d0o   = organizar_valores_propios(df_R5xiR5d0)
df_R5xiR5d10o  = organizar_valores_propios(df_R5xiR5d10)
df_R5xiR5d_10o = organizar_valores_propios(df_R5xiR5d_10)

# Cadena no lineal: E vs xi (R5 fijo)
df_xiR5_10d0o   = organizar_valores_propios(df_xiR5_10d0)
df_xiR5_10d10o  = organizar_valores_propios(df_xiR5_10d10)
df_xiR5_10d_10o = organizar_valores_propios(df_xiR5_10d_10)
df_xiR5_50d0o   = organizar_valores_propios(df_xiR5_50d0)
df_xiR5_50d10o  = organizar_valores_propios(df_xiR5_50d10)
df_xiR5_50d_10o = organizar_valores_propios(df_xiR5_50d_10)
# Cadena asimetrica: E vs F
df_AEvsF_R10_d_0o       = organizar_valores_propios(df_AEvsF_R10_d_0)
df_AEvsF_R10_d_10o      = organizar_valores_propios(df_AEvsF_R10_d_10)
df_AEvsF_R10_d_menos10o = organizar_valores_propios(df_AEvsF_R10_d_menos10)
df_AEvsF_R50_d_0o       = organizar_valores_propios(df_AEvsF_R50_d_0)
df_AEvsF_R50_d_10o      = organizar_valores_propios(df_AEvsF_R50_d_10)
df_AEvsF_R50_d_menos10o = organizar_valores_propios(df_AEvsF_R50_d_menos10)
print("Datos organizados OK")

## Step 5 — colour map

Change `COLORMAP` (for example `"viridis"`, `"cividis"`, `"magma"`) to recolour
all curves. The thesis uses `viridis`.


In [ ]:
def estilizar(ax):
    ax.grid(False)
    for s in ['top', 'right', 'left', 'bottom']:
        ax.spines[s].set_visible(True)
        ax.spines[s].set_linewidth(1.2)
        ax.spines[s].set_color('#333333')
    ax.tick_params(axis='both', which='both', length=0, colors='#333333', labelleft=True)
    return ax

def anotar_min(ax, df, k_cols, axis='x'):
    minimo = min(df[c].min() for c in k_cols)
    yl = ax.get_ylim(); xl = ax.get_xlim()
    y0 = yl[0] - 0.0012 if axis == 'x' else yl[0] - (yl[1] - yl[0]) * 0.04
    ax.text(xl[0], y0, f"Mín: {minimo:.5f}", fontsize=10, color='#333333',
            fontweight='bold', ha='left', va='top')
    return ax

from matplotlib.ticker import FuncFormatter

def _limpio(val):
    s = f"{val:.6f}".rstrip('0').rstrip('.')
    return s

def fmt_mev(v, pos=None):
    val = v * 1000
    if abs(val - round(val)) < 1e-6:
        return str(int(round(val)))
    return _limpio(val)

import matplotlib.image as mpimg

IMG_DIR_VON = "../figures/aux/von_neumann"

def recortar_orbital(img, pad=5):
    """Recorta el fondo dejando la pildora y su contorno negro.

    Usa el canal alpha (si la imagen tiene transparencia) para no cortar el
    contorno, y anade un margen `pad` para que el trazo no quede pegado al
    borde del subplot (que lo hacia ver punteado).
    """
    tiene_alpha = img.ndim == 3 and img.shape[2] == 4 and bool((img[:, :, 3] < 0.99).any())
    if tiene_alpha:
        mask = img[:, :, 3] > 0.01
    else:
        rgb = img[:, :, :3]
        corner = rgb[0, 0]
        mask = np.abs(rgb - corner).sum(axis=2) > 0.02

    ys, xs = np.where(mask)
    if len(ys) == 0:
        return img

    H, W = img.shape[:2]
    y0 = max(0, ys.min() - pad)
    y1 = min(H, ys.max() + 1 + pad)
    x0 = max(0, xs.min() - pad)
    x1 = min(W, xs.max() + 1 + pad)
    return img[y0:y1, x0:x1]


def recortar_puntos(img, pad=15):
    """Recorta a los circulos de los QD (excluye el rectangulo exterior).

    Detecta el contorno oscuro, descarta las lineas largas (bordes del
    rectangulo de la banda) y recorta al bounding box de los puntos + margen.
    """
    al = img[:, :, 3]
    op = al > 0.01
    if not op.any():
        return img
    lum = img[:, :, :3].mean(axis=2)
    dark = op & (lum < 0.35)
    H, W = dark.shape
    filas_largas = dark.sum(axis=1) > 0.5 * max(1, op.sum(axis=1).max())
    cols_largas = dark.sum(axis=0) > 0.5 * max(1, op.sum(axis=0).max())
    core = dark.copy()
    core[filas_largas, :] = False
    core[:, cols_largas] = False
    ys, xs = np.where(core)
    if len(ys) == 0:
        ys, xs = np.where(dark)
    y0 = max(0, ys.min() - pad)
    y1 = min(H, ys.max() + 1 + pad)
    x0 = max(0, xs.min() - pad)
    x1 = min(W, xs.max() + 1 + pad)
    return img[y0:y1, x0:x1]


# Part I — Uniform chain

Five dots of equal radius `R = 30 nm`, separated by `d = -10, 0, 10 nm`.


## Figure 1 — energy vs impurity position

The ground state stays near **-21.41 meV** for every `d`, close to the isolated
2D donor (-21.90 meV). The inset resolves the sub-meV oscillations and marks
three impurity positions.

**Look for:** the s-type ground state has minima at the centre of each dot;
the p-type first excited state has **maxima** there (see `docs/metodo.md`).


###  Energía vs $x_i$ (d = -10, 0, 10 nm) figura 1.

# Part II — Non-uniform chain

The first four dots keep `R = 30 nm`; the fifth dot has radius `R_5`, which
acts as a structural asymmetry. The next figures scan `R_5` from 10 to 60 nm.


## Figure 2 — energy vs `R_5` without impurity

Two features dominate: the formation of **sub-bands** (five coupled dots) and
the monotonic descent of the intruder level as `R_5` grows. At `R_5 = 30 nm`
the fifth level joins the first sub-band.


### Figura 2 — E vs $R_5$: sin impureza

## Figure 3 — energy vs `R_5`, impurity at `x_i = 0`

The donor sits at the centre of the chain. The Coulomb attraction dominates
and the spectrum is much less sensitive to `R_5` than in the empty chain.


### Figura 3 — E vs $R_5$: impureza $x_i=0$

## Figure 4 — energy vs `R_5`, impurity at `x_i = R_5`

The donor is placed at the **edge** of the fifth dot. Its level is now tied to
the size of QD5, so the crossing pattern changes qualitatively.


### Figura 4 — E vs $R_5$: impureza $x_i=R_5$

## Figure 5 — energy vs `R_5`, impurity at `x_i = R_1`

The donor is at the edge of the first dot. Compare with Figure 4: the chain is
not left-right symmetric, so the two edge positions are not equivalent.


### Figura 5 — E vs $R_5$: impureza $x_i=R_1$

## Figure 6 — energy vs `x_i`, `R_5 = 50 nm`

The fifth dot is **larger** than the others (acceptor-like). The impurity scan
shows how the extra space at the right end pulls the wavefunction.


## Energía vs $x_i$ $R_5=50$ nm figura 6

In [ ]:
# ============ LIMITES ============
xinf = [-132, -155, -180]
xsup = [171, 193, 215]


yinf = -0.022
ysup = 0.0035
# Límites de la gráfica a1
xi1_inf  = -130        # x inferior a1
xi1_sup  =  160        # x superior a1
yinfa1   = -0.02145    # y inferior a1
ysupa1   = -0.021    # y superior a1
# =================================

dfs = [globals()[v] for v in ['df_xiR5_50d_10o', 'df_xiR5_50d0o', 'df_xiR5_50d10o']]

k_cols = df1o.columns[1:11]
n = len(k_cols)
COLORMAP = "viridis"
colors = plt.get_cmap(COLORMAP)(np.linspace(0.15, 0.85, n))

casos = [
    {"df": dfs[0], "titulo": "a) d = -10 nm"},
    {"df": dfs[1], "titulo": "b) d = 0 nm"},
    {"df": dfs[2], "titulo": "c) d = 10 nm"},
]

fig = plt.figure(figsize=(12, 14), dpi=150)
gs = fig.add_gridspec(2, 2, height_ratios=[1.8, 1])

ax_a = fig.add_subplot(gs[0, :])
ax_b = fig.add_subplot(gs[1, 0])
ax_c = fig.add_subplot(gs[1, 1], sharey=ax_b)
axs = [ax_a, ax_b, ax_c]

etiquetas_estados = [rf"$\lambda_{{{i}}}$" for i in range(1, n + 1)]

for idx, caso in enumerate(casos):
    ax = axs[idx]
    df_actual = caso["df"]
    xi_nm = df_actual["X"] * 1e9

    for i, col in enumerate(k_cols, start=1):
        ax.plot(xi_nm, df_actual[col], color=colors[i-1], lw=2.5, alpha=0.95,
                solid_capstyle='round')

    # --- INSET EN PANEL A ---
    if idx == 0:
        ax_inset = ax.inset_axes([0.35, 0.18, 0.30, 0.32])
        first_state_col = k_cols[0]
        ax_inset.plot(xi_nm, df_actual[first_state_col],
                      color=colors[0], lw=1.5, alpha=0.95, solid_capstyle='round')
        ax_inset.set_title(r"$\mathbf{a_1}$) Oscilaciones primer estado", fontsize=10, pad=3)
        ax_inset.set_ylim(yinfa1, ysupa1)
        ax_inset.set_xlim(xi1_inf, xi1_sup)
        ax_inset.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_inset.yaxis.set_major_formatter(FuncFormatter(fmt_mev))
        ax_inset.spines[['top', 'right', 'bottom', 'left']].set_visible(True)
        ax_inset.spines[['top', 'right', 'bottom', 'left']].set_color('black')
        ax_inset.spines[['top', 'right', 'bottom', 'left']].set_linewidth(0.8)
        ax_inset.tick_params(axis='both', labelsize=10)

    ax.set_xlim(xinf[idx], xsup[idx])
    ax.set_ylim(yinf, ysup)

    ax.set_title(caso["titulo"], fontsize=14, pad=10)
    ax = estilizar(ax)
    ax.tick_params(axis='x', pad=7)

axs[0].set_ylabel(r"$\mathbf{Energía (meV)}$", fontsize=14)
axs[1].set_ylabel(r"$\mathbf{Energía (meV)}$", fontsize=14)
axs[2].set_ylabel(r"$\mathbf{Energía (meV)}$", fontsize=14)
for _ax in axs:
    _ax.yaxis.set_major_formatter(FuncFormatter(fmt_mev))

axs[0].set_xlabel(r"$\mathbf{x_i\ (nm)}$", fontsize=14)
axs[1].set_xlabel(r"$\mathbf{x_i\ (nm)}$", fontsize=14)
axs[2].set_xlabel(r"$\mathbf{x_i\ (nm)}$", fontsize=14)

# TEXTO DEL MÍNIMO ALINEADO VERTICALMENTE CON LA ETIQUETA DEL EJE X
fig.canvas.draw()
_res = fig.canvas.get_renderer()
from matplotlib.transforms import blended_transform_factory
for ax, caso in zip(axs, casos):
    df_actual = caso["df"]
    minimo = min(df_actual[c].min() for c in k_cols)
    _bbox = ax.xaxis.label.get_window_extent(_res)
    _y_fig = fig.transFigure.inverted().transform((_bbox.x0, (_bbox.y0 + _bbox.y1) / 2))[1]
    _t_min = blended_transform_factory(ax.transAxes, fig.transFigure)
    ax.text(0.0, _y_fig, f"Mín: {int(minimo*1000*100)/100:.2f}", fontsize=10, color='black',
            ha='left', va='center', transform=_t_min)

# LEYENDA GENERAL ARRIBA A LA DERECHA
lines = [plt.Line2D([0], [0], color=c, lw=2.5) for c in colors]
fig.legend(lines, etiquetas_estados,
           title="Estados",
           title_fontsize='10',
           loc='upper left',
           bbox_to_anchor=(0.91, 0.88),
           frameon=True,
           facecolor='white',
           edgecolor='gray')


fig.suptitle(r'Energia vs $\mathbf{x_i}$ con $\mathbf{R_5=50}$ nm', fontsize=18, weight='semibold', y=0.96)

plt.subplots_adjust(top=0.90, bottom=0.14, left=0.08, right=0.90, hspace=0.18, wspace=0.32)

plt.savefig('../figures/EvsxiR5_50.png', dpi=300, bbox_inches='tight')
plt.show()


## Figure 7 — energy vs `x_i`, `R_5 = 10 nm`

The fifth dot is **smaller** (donor-like): it gives the electron to the other
four dots. The fifth oscillation of the ground state is suppressed.


## Energía vs $x_i$ $R_5=10$ nm figura 7

In [ ]:
# ============ LIMITES ============
xinf = [-132, -155, -180]
xsup = [90, 113, 132]
yinf = -0.022
ysup = 0.005
# Límites de la gráfica a1
xi1_inf  = -120        # x inferior a1
xi1_sup  =  75        # x superior a1
yinfa1   = -0.0215     # y inferior a1
ysupa1   = -0.0210     # y superior a1
# =================================

dfs = [globals()[v] for v in ['df_xiR5_10d_10o', 'df_xiR5_10d0o', 'df_xiR5_10d10o']]

k_cols = df1o.columns[1:11]
n = len(k_cols)
COLORMAP = "viridis"
colors = plt.get_cmap(COLORMAP)(np.linspace(0.15, 0.85, n))

casos = [
    {"df": dfs[0], "titulo": "a) d = -10 nm"},
    {"df": dfs[1], "titulo": "b) d = 0 nm"},
    {"df": dfs[2], "titulo": "c) d = 10 nm"},
]

fig = plt.figure(figsize=(12, 14), dpi=150)
gs = fig.add_gridspec(2, 2, height_ratios=[1.8, 1])

ax_a = fig.add_subplot(gs[0, :])
ax_b = fig.add_subplot(gs[1, 0])
ax_c = fig.add_subplot(gs[1, 1], sharey=ax_b)
axs = [ax_a, ax_b, ax_c]

etiquetas_estados = [rf"$\lambda_{{{i}}}$" for i in range(1, n + 1)]

for idx, caso in enumerate(casos):
    ax = axs[idx]
    df_actual = caso["df"]
    xi_nm = df_actual["X"] * 1e9

    for i, col in enumerate(k_cols, start=1):
        ax.plot(xi_nm, df_actual[col], color=colors[i-1], lw=2.5, alpha=0.95,
                solid_capstyle='round')

    # --- INSET EN PANEL A ---
    if idx == 0:
        ax_inset = ax.inset_axes([0.35, 0.18, 0.30, 0.32])
        first_state_col = k_cols[0]
        ax_inset.plot(xi_nm, df_actual[first_state_col],
                      color=colors[0], lw=1.5, alpha=0.95, solid_capstyle='round')
        ax_inset.set_title(r"$\mathbf{a_1}$) Oscilaciones primer estado", fontsize=10, pad=3)
        ax_inset.set_ylim(yinfa1, ysupa1)
        ax_inset.set_xlim(xi1_inf, xi1_sup)
        ax_inset.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax_inset.yaxis.set_major_formatter(FuncFormatter(fmt_mev))
        ax_inset.spines[['top', 'right', 'bottom', 'left']].set_visible(True)
        ax_inset.spines[['top', 'right', 'bottom', 'left']].set_color('black')
        ax_inset.spines[['top', 'right', 'bottom', 'left']].set_linewidth(0.8)
        ax_inset.tick_params(axis='both', labelsize=10)

    ax.set_xlim(xinf[idx], xsup[idx])
    ax.set_ylim(yinf, ysup)

    ax.set_title(caso["titulo"], fontsize=14, pad=10)
    ax = estilizar(ax)
    ax.tick_params(axis='x', pad=7)

axs[0].set_ylabel(r"$\mathbf{Energía (meV)}$", fontsize=14)
axs[1].set_ylabel(r"$\mathbf{Energía (meV)}$", fontsize=14)
axs[2].set_ylabel(r"$\mathbf{Energía (meV)}$", fontsize=14)
for _ax in axs:
    _ax.yaxis.set_major_formatter(FuncFormatter(fmt_mev))

axs[0].set_xlabel(r"$\mathbf{x_i\ (nm)}$", fontsize=14)
axs[1].set_xlabel(r"$\mathbf{x_i\ (nm)}$", fontsize=14)
axs[2].set_xlabel(r"$\mathbf{x_i\ (nm)}$", fontsize=14)

# TEXTO DEL MÍNIMO ALINEADO VERTICALMENTE CON LA ETIQUETA DEL EJE X
fig.canvas.draw()
_res = fig.canvas.get_renderer()
from matplotlib.transforms import blended_transform_factory
for ax, caso in zip(axs, casos):
    df_actual = caso["df"]
    minimo = min(df_actual[c].min() for c in k_cols)
    _bbox = ax.xaxis.label.get_window_extent(_res)
    _y_fig = fig.transFigure.inverted().transform((_bbox.x0, (_bbox.y0 + _bbox.y1) / 2))[1]
    _t_min = blended_transform_factory(ax.transAxes, fig.transFigure)
    ax.text(0.0, _y_fig, f"Mín: {int(minimo*1000*100)/100:.2f}", fontsize=10, color='black',
            ha='left', va='center', transform=_t_min)

# LEYENDA GENERAL ARRIBA A LA DERECHA
lines = [plt.Line2D([0], [0], color=c, lw=2.5) for c in colors]
fig.legend(lines, etiquetas_estados,
           title="Estados",
           title_fontsize='10',
           loc='upper left',
           bbox_to_anchor=(0.91, 0.88),
           frameon=True,
           facecolor='white',
           edgecolor='gray')


fig.suptitle(r'Energia vs $\mathbf{x_i}$ con $\mathbf{R_5=10}$ nm', fontsize=18, weight='semibold', y=0.96)

plt.subplots_adjust(top=0.90, bottom=0.14, left=0.08, right=0.90, hspace=0.18, wspace=0.32)

plt.savefig('../figures/EvsxiR5_10.png', dpi=300, bbox_inches='tight')
plt.show()


# Part III — Electric field and induced dipole

The field is applied along the chain. The induced dipole is obtained from the
slope of the ground-state energy, `p = -dE/dF`. Each figure fits two straight
lines on either side of the anticrossing and reports the plateau values
`p_1` and `p_2`.


## Uniform chain — field response

The three panels use `d = -10, 0, 10 nm` and the same fitting window
(`-1.1 <= F <= 2.0 kV/cm`). The vertical dotted line is the critical field.


## Figure 8 — energy vs electric field (uniform chain)

The levels bend and the ground state changes character at the anticrossing.

**Counterintuitive:** the critical field is **inversely proportional to `d`**
(1.2 → 1.0 → 0.8 kV/cm). A more separated chain needs *less* field to
delocalize the electron, because the electron is already more confined.


In [ ]:


# ==========================================
# FIGURA A - Energia vs F - paneles a) (d=-10, 0, 10)
# ==========================================
# ============ LIMITES ============
xinf = [-1, -1, -1]                 # F (kV/cm) inferior, por caso: d=-10, d=0, d=10
xsup = [3.0, 2.7, 2.5]                 # F (kV/cm) superior, por caso
yinf = [-0.03, -0.03, -0.03]        # Energia (eV) inferior, por caso
ysup = [0.004, 0.004, 0.005]        # Energia (eV) superior, por caso
# ==========================================


casos_a = [
    {"df": df3Fo, "titulo": "a) d = -10 nm"},
    {"df": df1Fo, "titulo": "b) d = 0 nm"},
    {"df": df2_2Fo, "titulo": "c) d = 10 nm"},
]

COLORMAP = "viridis"

fig, axs = plt.subplots(1, 3, figsize=(16, 5.0), dpi=150)

for idx, caso in enumerate(casos_a):
    ax = axs[idx]
    df_actual = caso["df"]
    xi_nm = df_actual["X"]
    k_act = df_actual.columns[1:]
    n_act = len(k_act)
    colors_act = plt.get_cmap(COLORMAP)(np.linspace(0.15, 0.85, n_act))
    for i, col in enumerate(k_act, start=1):
        ax.plot(xi_nm, df_actual[col], color=colors_act[i-1], lw=2.5, alpha=0.95,
                linestyle=':', solid_capstyle='round')
    ax.set_xlabel(r"$\mathbf{F\ \left(\frac{kV}{cm}\right)}$")
    ax.set_ylabel(r"$\mathbf{Energía\ (meV)}$")
    ax.yaxis.set_major_formatter(FuncFormatter(fmt_mev))
    ax.set_xlim(xinf[idx], xsup[idx])
    ax.set_ylim(yinf[idx], ysup[idx])
    ax.set_title(caso["titulo"], fontsize=14, pad=10)
    ax = estilizar(ax)
    ax.tick_params(axis='x', pad=7)


fig.suptitle('Energía vs campo eléctrico (cadena uniforme)', fontsize=18, weight='semibold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('../figures/EvsF_uniforme.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 9 — induced dipole vs electric field (uniform chain)

The dipole jumps from `p_1` to `p_2` at the anticrossing. In the symmetric
chain `|p_1| < |p_2|`; asymmetry can invert this relation (see the next
figures and `docs/metodo.md`).

The lower panels show the probability densities before and after the jump.


In [ ]:
# ==========================================
# FIGURA B - Momento dipolar inducido - paneles b) (d=-10, 0, 10)
# ==========================================
# ============ LIMITES ============
DELTA   = 0.8   # medio-ancho de la ventana X (F) alrededor del quiebre (centra la linea vertical)
# ==========================================

from matplotlib.ticker import MaxNLocator
import matplotlib.image as mpimg

IMG_DIR = "../figures/aux/momento"

def pendiente_forzada(x, y, x0, y0):
    return np.sum((x - x0) * (y - y0)) / np.sum((x - x0)**2)

casos_b = [
    {"df": df3Fo,   "titulo": "a) d = -10 nm", "img_iz": "iz_d_-10.png", "img_d": "d_d_-10.png", "etq": "a"},
    {"df": df1Fo,   "titulo": "b) d = 0 nm",   "img_iz": "i_d=0.png",    "img_d": "d_d=0.png",    "etq": "b"},
    {"df": df2_2Fo, "titulo": "c) d = 10 nm",  "img_iz": "i_d=10.png",   "img_d": "d_d=10.png",   "etq": "c"},
]

fig = plt.figure(figsize=(16, 10), dpi=150)
gs = fig.add_gridspec(2, 6, height_ratios=[1.7, 0.24], hspace=0.30, wspace=0.35)

for idx, caso in enumerate(casos_b):
    df_actual = caso["df"]
    F_all = np.array(df_actual['X'])
    col = df_actual.columns[1]
    E = np.array(df_actual[col])

    # --- Analisis: ajuste dos rectas (primer valor propio) ---
    mask = (F_all >= -1.1) & (F_all <= 2.0)
    Fm, Em = F_all[mask], E[mask]
    idx_local = np.argmax(Em)
    F_quiebre = Fm[idx_local]
    E_quiebre = Em[idx_local]
    F1, E1 = Fm[:idx_local+1], Em[:idx_local+1]
    F2, E2 = Fm[idx_local:], Em[idx_local:]
    m1 = pendiente_forzada(F1, E1, F_quiebre, E_quiebre)
    m2 = pendiente_forzada(F2, E2, F_quiebre, E_quiebre)
    # --- Momento dipolar inducido p = -dE/dF ---
    # Valores de referencia de la tabla (ya negados, en meV/(kV/cm))
    _p_tabla = {'-10': (-9.8934, 10.4221), '0': (-11.9967, 12.5312), '10': (-14.9987, 15.2961)}
    _dkey = caso["titulo"].split("d = ")[1].replace(" nm", "")
    p1, p2 = _p_tabla[_dkey]
    print(f'{caso["titulo"]} | {col} | p1={p1:.6f} | p2={p2:.6f} | F_quiebre={F_quiebre:.3f}')

    # --- Rectas del momento inducido ---
    pend1 = np.full_like(F_all, np.nan, dtype=float)
    pend2 = np.full_like(F_all, np.nan, dtype=float)
    pend1[np.isin(F_all, F1)] = p1
    pend2[np.isin(F_all, F2)] = p2

    ax = fig.add_subplot(gs[0, 2*idx:2*idx+2])
    ax.plot(F_all, pend1, color='black', linestyle='-', lw=3)
    ax.plot(F_all, pend2, color='black', linestyle='--', lw=3)
    ax.axvline(F_quiebre, color='#4FC3F7', linestyle=':', lw=3)

    span = (p2 - p1) / 0.44
    ax.text(0.06, p1 + 0.02*span, rf'$\mathbf{{p_1 = {p1:.2f}}}$ u.a.',
            transform=ax.get_yaxis_transform(), fontsize=16, color='black',
            ha='left', va='bottom')
    ax.text(0.94, p2 + 0.02*span, rf'$\mathbf{{p_2 = {p2:.2f}}}$ u.a.',
            transform=ax.get_yaxis_transform(), fontsize=16, color='black',
            ha='right', va='bottom')
    ax.text(F_quiebre + 0.05, 0.45, rf'$\mathbf{{{F_quiebre:.3f}\ \frac{{kV}}{{cm}}}}$',
            transform=ax.get_xaxis_transform(), color='#2E75B6', fontsize=18,
            ha='left', va='center')
    ax.set_xlabel(r"$\mathbf{F\ \left(\frac{kV}{cm}\right)}$")
    if idx == 0:
        ax.set_ylabel(r"$\mathbf{P\ (u.a.)}$")
    # --- Rango X re-centrado al quiebre (linea vertical punteada al centro) ---
    ax.set_xlim(F_quiebre - DELTA, F_quiebre + DELTA)

    # --- Limite Y normalizado: p1 al 72% de la altura, p2 al 28% ---
    ylo = p1 - 0.28 * span
    yhi = p2 + 0.28 * span
    ax.set_ylim(ylo, yhi)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6, steps=[1, 2, 5, 10]))

    ax = estilizar(ax)
    ax.tick_params(axis='x', pad=7)
    ax.set_title(caso["titulo"], fontsize=14, pad=10)

    # --- Orbitales abajo: dos recuadros lado a lado ---
    orbitales = [(caso["img_d"], rf"$\mathbf{{{caso['etq']}_1}}$)"), (caso["img_iz"], rf"$\mathbf{{{caso['etq']}_2}}$)")]
    for j, (img_nombre, etiqueta) in enumerate(orbitales):
        axo = fig.add_subplot(gs[1, 2*idx + j])
        axo.imshow(recortar_orbital(mpimg.imread(f"{IMG_DIR}/{img_nombre}")), aspect='auto')
        axo.axis('off')
        axo.set_title(etiqueta, fontsize=14, pad=2, loc='center')
        for spine in axo.spines.values():
            spine.set_visible(True)
            spine.set_color('#333333')
            spine.set_linewidth(1.2)

fig.suptitle('Momento dipolar inducido vs campo eléctrico (cadena uniforme)', fontsize=18, weight='semibold')
plt.tight_layout()
plt.savefig('../figures/Momentos_dipolares_uniforme.png', dpi=300, bbox_inches='tight')
plt.show()

## Figure 10 — energy vs electric field (asymmetric chain, `R = 10 nm`)

A reduced fifth dot lowers the magnitude of the induced dipole. The two-branch
structure survives, but the plateau values are smaller than in the uniform
chain.


In [ ]:
def plot_energy_vs_field(casos, outfile, f_max=2.5):
    """Three panels: energy vs F for d = -10, 0, 10 nm."""
    fig, axs = plt.subplots(1, 3, figsize=(16, 5), dpi=150, sharey=True)
    for ax, (label, df) in zip(axs, casos):
        f = df["X"]
        for col in df.columns[1:]:
            ax.plot(f, df[col], lw=1.8)
        ax.set_xlim(-1, f_max)
        ax.set_ylim(-0.03, 0.005)
        ax.set_title(label, fontsize=14, pad=10)
        ax.set_xlabel(r"$\mathbf{F\ (kV/cm)}$")
        ax = estilizar(ax)
    axs[0].set_ylabel("Energy (eV)")
    fig.suptitle("Energy vs electric field (asymmetric chain)", fontsize=18, weight="semibold")
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()


plot_energy_vs_field([
    ("d = -10 nm", df_AEvsF_R10_d_menos10o),
    ("d = 0 nm",   df_AEvsF_R10_d_0o),
    ("d = 10 nm",  df_AEvsF_R10_d_10o),
], "../figures/EvsF_NO_UNIF_R=10.png")

## Figure 11 — induced dipole vs field (asymmetric chain, `R = 10 nm`)

The plateaus are computed from the data with the same forced-slope fit used in
the uniform case. No hard-coded values are needed.


In [ ]:
def plot_dipole_panels(casos, outfile):
    """Three panels: p = -dE/dF, with the orbital insets below."""
    fig = plt.figure(figsize=(16, 10), dpi=150)
    gs = fig.add_gridspec(2, 6, height_ratios=[1.7, 0.24], hspace=0.30, wspace=0.35)

    for idx, caso in enumerate(casos):
        df = caso["df"]
        f_all = np.asarray(df["X"], dtype=float)
        e_all = np.asarray(df.iloc[:, 1], dtype=float)

        mask = (f_all >= -1.1) & (f_all <= 2.0)
        fm, em = f_all[mask], e_all[mask]
        i0 = int(np.argmax(em))
        f0, e0 = fm[i0], em[i0]
        f1, e1 = fm[:i0 + 1], em[:i0 + 1]
        f2, e2 = fm[i0:], em[i0:]
        p1 = -pendiente_forzada(f1, e1, f0, e0) * 1000
        p2 = -pendiente_forzada(f2, e2, f0, e0) * 1000

        pend1 = np.full_like(f_all, np.nan, dtype=float)
        pend2 = np.full_like(f_all, np.nan, dtype=float)
        pend1[np.isin(f_all, f1)] = p1
        pend2[np.isin(f_all, f2)] = p2

        ax = fig.add_subplot(gs[0, 2 * idx:2 * idx + 2])
        ax.plot(f_all, pend1, color="black", linestyle="-", lw=3)
        ax.plot(f_all, pend2, color="black", linestyle="--", lw=3)
        ax.axvline(f0, color="#4FC3F7", linestyle=":", lw=3)

        span = (p2 - p1) / 0.44
        ax.text(0.06, p1 + 0.02 * span, rf"$\mathbf{{p_1 = {p1:.3f}}}$",
                transform=ax.get_yaxis_transform(), fontsize=16, ha="left", va="bottom")
        ax.text(0.94, p2 + 0.02 * span, rf"$\mathbf{{p_2 = {p2:.3f}}}$",
                transform=ax.get_yaxis_transform(), fontsize=16, ha="right", va="bottom")
        ax.text(f0 + 0.05, 0.45, rf"$\mathbf{{{f0:.3f}\ kV/cm}}$",
                transform=ax.get_xaxis_transform(), color="#2E75B6", fontsize=16)
        ax.set_xlim(f0 - 0.8, f0 + 0.8)
        ax.set_ylim(p1 - 0.28 * span, p2 + 0.28 * span)
        ax.set_xlabel(r"$\mathbf{F\ (kV/cm)}$")
        if idx == 0:
            ax.set_ylabel("p = -dE/dF  (meV/(kV/cm))")
        ax.set_title(caso["titulo"], fontsize=14, pad=10)
        ax = estilizar(ax)

        for j, (img_name, tag) in enumerate(
            [(caso["img_d"], f"{caso['etq']}_1"), (caso["img_iz"], f"{caso['etq']}_2")]
        ):
            axo = fig.add_subplot(gs[1, 2 * idx + j])
            axo.imshow(recortar_orbital(mpimg.imread(f"{IMG_DIR}/{img_name}")), aspect="auto")
            axo.axis("off")
            axo.set_title(tag, fontsize=13, pad=2)
            for spine in axo.spines.values():
                spine.set_visible(True)
                spine.set_color("#333333")
                spine.set_linewidth(1.2)

    fig.suptitle("Induced dipole vs electric field (asymmetric chain)", fontsize=18, weight="semibold")
    plt.tight_layout()
    plt.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.show()


plot_dipole_panels([
    {"df": df_AEvsF_R10_d_menos10o, "titulo": "a) d = -10 nm", "img_iz": "i_R10_-10.png", "img_d": "d_R10_-10.png", "etq": "a"},
    {"df": df_AEvsF_R10_d_0o,       "titulo": "b) d = 0 nm",   "img_iz": "i_R10_0.png",   "img_d": "d_R10_0.png",   "etq": "b"},
    {"df": df_AEvsF_R10_d_10o,      "titulo": "c) d = 10 nm",  "img_iz": "i_R10_10.png",  "img_d": "d_R10_10.png",  "etq": "c"},
], "../figures/Momentos_dipolares_NO_UNIF_R=10.png")

## Figure 12 — energy vs electric field (asymmetric chain, `R = 50 nm`)

An enlarged fifth dot increases the dipole magnitude. The critical field still
follows the inverse-`d` trend.


In [ ]:
plot_energy_vs_field([
    ("d = -10 nm", df_AEvsF_R50_d_menos10o),
    ("d = 0 nm",   df_AEvsF_R50_d_0o),
    ("d = 10 nm",  df_AEvsF_R50_d_10o),
], "../figures/EvsF_NO_UNIF_R=50.png")

## Figure 13 — induced dipole vs field (asymmetric chain, `R = 50 nm`)

For `R_5 = 50 nm` the hierarchy can become `|p_1| > |p_2|`: the larger fifth
dot makes the distance from the chain centre to the right longer than to the
left.


In [ ]:
plot_dipole_panels([
    {"df": df_AEvsF_R50_d_menos10o, "titulo": "a) d = -10 nm", "img_iz": "i_R50-10.png", "img_d": "d_R50-10.png", "etq": "p"},
    {"df": df_AEvsF_R50_d_0o,       "titulo": "b) d = 0 nm",   "img_iz": "i_R50_0.png",  "img_d": "d_R50_0.png",  "etq": "p"},
    {"df": df_AEvsF_R50_d_10o,      "titulo": "c) d = 10 nm",  "img_iz": "i_R50_10.png", "img_d": "d_R50_10.png", "etq": "p"},
], "../figures/Momentos_dipolares_NO_UNIF_R=50.png")

## Key takeaways

- The ground state is pinned by the donor near **-21.41 meV**, almost
  independent of `d` and of the chain geometry.
- The **symmetry rule** explains the spectrum: s-type states minimize at dot
  centres, p-type states maximize there and minimize at the bottlenecks.
- The critical field is **inversely proportional to `d`**.
- Structural asymmetry (QD5 size) can turn the fifth dot into a donor or an
  acceptor and can invert the `|p_1|` / `|p_2|` hierarchy.

Continue with `01_data_pipeline.ipynb` for the data conventions and
`03_dipole_analysis.ipynb` for a compact derivation of `p_1`, `p_2` and
`F_critical`.
